# Kaggle Template - WESAD Stress Detection (ML + DL)

Notebook nay de chay benchmark WESAD theo LOSO cho:
- ML (`src/training.py`)
- DL (`src/dl_training.py`) voi `cnn1d`, `unet1d`, `resnet1d`

Huong dan nhanh:
1. Add dataset WESAD vao Kaggle Input (co cac thu muc `S2...S17`).
2. Chon cach nap source code (`SOURCE_MODE`) o cell Config.
3. Chay lan luot tu tren xuong duoi.


In [ ]:
import os
import sys
import json
import shutil
import subprocess
import importlib.util
from pathlib import Path
from datetime import datetime

def run_cmd(cmd, cwd=None):
    if isinstance(cmd, str):
        printable = cmd
        shell = True
    else:
        printable = " ".join(cmd)
        shell = False
    print(f"\n$ {printable}")
    subprocess.check_call(cmd, cwd=cwd, shell=shell)

def has_project_root(path: Path) -> bool:
    return (path / "src" / "training.py").exists() and (path / "src" / "dl_training.py").exists()

def latest_files(directory: Path, pattern: str, top_k: int = 5):
    files = sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime)
    return files[-top_k:]


In [ ]:
# =========================
# Config
# =========================

# Cac gia tri hop le: "auto", "git", "input", "current"
SOURCE_MODE = "auto"

# Neu dung git
REPO_URL = "https://github.com/xanhhangreal/TotNghiep.git"
REPO_BRANCH = "main"

# Neu dung Kaggle input (code dataset)
CODE_INPUT_HINT = None
# Vi du:
# CODE_INPUT_HINT = "/kaggle/input/totnghiep-code/TotNghiep"

# Noi copy/clone code vao
PROJECT_DIR = Path("/kaggle/working/TotNghiep")

# WESAD
WESAD_INPUT_HINT = None
# Vi du:
# WESAD_INPUT_HINT = "/kaggle/input/wesad/WESAD"

# Chay gi
RUN_ML = True
RUN_DL = True

# Protocol
RUN_PAPER_PROTOCOL = True   # 60s window, 0.25s step
DEVICE_MODE = "both"       # wrist | chest | both
RUN_SUBJECTS = None         # None de chay full 15 subjects

# Window/step thu cong (chi dung khi RUN_PAPER_PROTOCOL=False)
ML_WINDOW = 60.0
ML_STEP = 30.0
DL_WINDOW = 60.0
DL_STEP = 30.0

# DL options
DL_ARCHES = ["cnn1d", "unet1d", "resnet1d"]
DL_CLASS_MODES = ["binary", "3class"]
DL_EPOCHS = 100
DL_BATCH_SIZE = 64
DL_LR = 1e-3

# Neu True: bo qua run neu da co file json ket qua tuong ung
SKIP_IF_RESULT_EXISTS = False

print("Config loaded.")


In [ ]:
# =========================
# Resolve project code path
# =========================

def detect_code_input_dir(hint=None):
    candidates = []
    if hint:
        candidates.append(Path(hint))

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for ds in sorted(input_root.glob("*")):
            if not ds.is_dir():
                continue
            candidates.append(ds)
            for child in ds.glob("*"):
                if child.is_dir():
                    candidates.append(child)

    seen = set()
    for c in candidates:
        c = c.resolve()
        if c in seen:
            continue
        seen.add(c)
        if has_project_root(c):
            return c
    return None

cwd = Path.cwd().resolve()
project_root = None

if SOURCE_MODE == "current":
    if not has_project_root(cwd):
        raise FileNotFoundError(f"Current directory is not project root: {cwd}")
    project_root = cwd

elif SOURCE_MODE == "input":
    code_input = detect_code_input_dir(CODE_INPUT_HINT)
    if code_input is None:
        raise FileNotFoundError("Cannot find project code in /kaggle/input. Set CODE_INPUT_HINT.")
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    shutil.copytree(code_input, PROJECT_DIR)
    project_root = PROJECT_DIR

elif SOURCE_MODE == "git":
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    run_cmd(["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(PROJECT_DIR)])
    project_root = PROJECT_DIR

elif SOURCE_MODE == "auto":
    if has_project_root(cwd):
        project_root = cwd
    else:
        code_input = detect_code_input_dir(CODE_INPUT_HINT)
        if code_input is not None:
            if PROJECT_DIR.exists():
                shutil.rmtree(PROJECT_DIR)
            shutil.copytree(code_input, PROJECT_DIR)
            project_root = PROJECT_DIR
        else:
            if PROJECT_DIR.exists():
                shutil.rmtree(PROJECT_DIR)
            run_cmd(["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(PROJECT_DIR)])
            project_root = PROJECT_DIR
else:
    raise ValueError("SOURCE_MODE must be one of: auto, git, input, current")

os.chdir(project_root)
PROJECT_DIR = Path.cwd().resolve()
print("Project root:", PROJECT_DIR)
print("Exists training.py:", (PROJECT_DIR / "src" / "training.py").exists())
print("Exists dl_training.py:", (PROJECT_DIR / "src" / "dl_training.py").exists())


In [ ]:
# =========================
# Environment check (install missing core deps only)
# =========================

required_import_to_pkg = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "sklearn": "scikit-learn",
    "joblib": "joblib",
    "torch": "torch"
}

missing = [pkg for imp, pkg in required_import_to_pkg.items() if importlib.util.find_spec(imp) is None]
if missing:
    print("Missing packages:", missing)
    run_cmd([sys.executable, "-m", "pip", "install", "-q"] + missing)
else:
    print("Core dependencies are available.")

import numpy as np
import torch
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# =========================
# Detect WESAD root
# =========================

SUBJECTS = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]

def is_wesad_root(path: Path) -> bool:
    return all((path / f"S{sid}" / f"S{sid}.pkl").exists() for sid in SUBJECTS)

def detect_wesad_root(hint=None):
    candidates = []
    if hint:
        candidates.append(Path(hint))

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for ds in sorted(input_root.glob("*")):
            if not ds.is_dir():
                continue
            candidates.append(ds)
            candidates.append(ds / "WESAD")
            for child in ds.glob("*"):
                if child.is_dir():
                    candidates.append(child)
                    candidates.append(child / "WESAD")

    seen = set()
    for c in candidates:
        c = c.resolve()
        if c in seen:
            continue
        seen.add(c)
        if c.exists() and c.is_dir() and is_wesad_root(c):
            return c

    # Fallback recursive search
    if input_root.exists():
        for s2 in input_root.rglob("S2.pkl"):
            if s2.parent.name != "S2":
                continue
            root = s2.parent.parent
            if is_wesad_root(root):
                return root

    top = [p.name for p in sorted(input_root.glob("*"))] if input_root.exists() else []
    raise FileNotFoundError(
        "Could not detect WESAD root. Set WESAD_INPUT_HINT manually. "
        f"Top-level /kaggle/input entries: {top}"
    )

WESAD_DIR = detect_wesad_root(WESAD_INPUT_HINT)
print("WESAD_DIR:", WESAD_DIR)

missing = [sid for sid in SUBJECTS if not (WESAD_DIR / f"S{sid}" / f"S{sid}.pkl").exists()]
print("Missing subjects:", missing)
if missing:
    raise RuntimeError(f"WESAD missing subjects: {missing}")


In [ ]:
# =========================
# Optional: ML LOSO benchmark
# =========================

if RUN_ML:
    for n_cls in [2, 3]:
        cmd = [
            sys.executable, "-u", "src/training.py",
            "--approach", "loso",
            "--device", DEVICE_MODE,
            "--n-classes", str(n_cls),
            "--wesad-dir", str(WESAD_DIR),
        ]
        if RUN_SUBJECTS:
            cmd += ["--subjects"] + [str(s) for s in RUN_SUBJECTS]
        if RUN_PAPER_PROTOCOL:
            cmd.append("--paper-protocol")
        else:
            cmd += ["--window", str(ML_WINDOW), "--step", str(ML_STEP)]
        run_cmd(cmd, cwd=PROJECT_DIR)
else:
    print("RUN_ML=False -> skip ML block")


In [ ]:
# =========================
# DL LOSO benchmark
# =========================

def has_result_for(arch: str, cls_mode: str):
    cls_tag = "2cls" if cls_mode == "binary" else "3cls"
    pat = f"dl_loso_{arch}_{cls_tag}_*.json"
    files = list((PROJECT_DIR / "results").glob(pat))
    return len(files) > 0

if RUN_DL:
    for cls_mode in DL_CLASS_MODES:
        for arch in DL_ARCHES:
            if SKIP_IF_RESULT_EXISTS and has_result_for(arch, cls_mode):
                print(f"Skip arch={arch}, classes={cls_mode} (result exists)")
                continue

            print("\n" + "=" * 80)
            print(f"Running arch={arch}, classes={cls_mode}")
            print("=" * 80)

            cmd = [
                sys.executable, "-u", "src/dl_training.py",
                "--arch", arch,
                "--classes", cls_mode,
                "--approach", "loso",
                "--device", DEVICE_MODE,
                "--wesad-dir", str(WESAD_DIR),
                "--epochs", str(DL_EPOCHS),
                "--batch-size", str(DL_BATCH_SIZE),
                "--lr", str(DL_LR),
            ]
            if RUN_SUBJECTS:
                cmd += ["--subjects"] + [str(s) for s in RUN_SUBJECTS]
            if RUN_PAPER_PROTOCOL:
                cmd.append("--paper-protocol")
            else:
                cmd += ["--window", str(DL_WINDOW), "--step", str(DL_STEP)]

            run_cmd(cmd, cwd=PROJECT_DIR)
else:
    print("RUN_DL=False -> skip DL block")


In [ ]:
# =========================
# Summary table from JSON results
# =========================

import pandas as pd

results_dir = PROJECT_DIR / "results"
json_files = sorted(results_dir.glob("*.json"), key=lambda p: p.stat().st_mtime)
print(f"Total result files: {len(json_files)}")

records = []

for p in json_files:
    name = p.name
    try:
        data = json.loads(p.read_text(encoding="utf-8"))
    except Exception:
        continue

    # ML LOSO
    if name.startswith("loso_"):
        for model_name, m in data.items():
            if isinstance(m, dict) and "accuracy_mean" in m:
                records.append({
                    "file": name,
                    "family": "ML",
                    "model": model_name,
                    "classes": "unknown",
                    "accuracy_mean": m.get("accuracy_mean"),
                    "accuracy_std": m.get("accuracy_std"),
                    "f1_mean": m.get("f1_mean"),
                    "f1_std": m.get("f1_std"),
                })

    # DL LOSO
    if name.startswith("dl_loso_") and isinstance(data, dict):
        records.append({
            "file": name,
            "family": "DL",
            "model": data.get("arch", "unknown"),
            "classes": data.get("n_classes", "unknown"),
            "accuracy_mean": data.get("accuracy_mean"),
            "accuracy_std": data.get("accuracy_std"),
            "f1_mean": data.get("f1_mean"),
            "f1_std": data.get("f1_std"),
        })

df = pd.DataFrame(records)
if len(df) == 0:
    print("No parsable LOSO result files found.")
else:
    df = df.sort_values(["family", "model", "classes", "file"]).reset_index(drop=True)
    display(df.tail(30))

    latest_per_group = (
        df.groupby(["family", "model", "classes"], as_index=False)
          .tail(1)
          .sort_values(["family", "classes", "accuracy_mean"], ascending=[True, True, False])
          .reset_index(drop=True)
    )

    print("\nLatest per model/class:")
    display(latest_per_group)

    summary_csv = results_dir / "kaggle_summary_latest.csv"
    latest_per_group.to_csv(summary_csv, index=False)
    print("Saved:", summary_csv)


In [ ]:
# =========================
# Export outputs for download
# =========================

from zipfile import ZipFile

export_dir = Path("/kaggle/working/wesad_export")
if export_dir.exists():
    shutil.rmtree(export_dir)
export_dir.mkdir(parents=True, exist_ok=True)

for p in sorted((PROJECT_DIR / "results").glob("*.json")):
    shutil.copy2(p, export_dir / p.name)

summary_csv = PROJECT_DIR / "results" / "kaggle_summary_latest.csv"
if summary_csv.exists():
    shutil.copy2(summary_csv, export_dir / summary_csv.name)

zip_path = Path("/kaggle/working/wesad_results.zip")
if zip_path.exists():
    zip_path.unlink()

with ZipFile(zip_path, "w") as zf:
    for p in sorted(export_dir.glob("*")):
        if p.is_file():
            zf.write(p, arcname=p.name)

print("Export folder:", export_dir)
print("Zip file:", zip_path)
print("Download from Kaggle Output panel.")
